# aule — Divergent colormap normalizations

When most of a divergent field's values cluster near zero (background noise) but a handful of extreme anomalies matter most, a plain linear colormap normalization wastes most of its color range on the noise and saturates extremes to the same dark color, making it hard to tell a strong anomaly from an even stronger one.

This notebook compares aule's normalization helpers (`aule.plots._style.power_norm`, `symlog_norm`, `asymmetric_twoslope_norm`, and the `resolve_diverging_norm` dispatcher used internally by `plot_bias_map`, `plot_error_map`, and `plot_field_comparison`) to make extreme values stand out.

In [ ]:
!pip install aule

## Synthetic test data

Mostly small Gaussian noise around zero, with a handful of strong positive/negative Gaussian-blob anomalies layered on top — a stand-in for something like a bias or anomaly map where small-scale noise should stay calm but real signal should pop out.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

n = 500
data = np.random.normal(0, 0.05, (n, n))

for _ in range(20):
    x0, y0 = np.random.randint(0, n, size=2)
    amplitude = np.random.choice([-1, 1]) * np.random.uniform(0.5, 1.0)
    sigma = np.random.uniform(10, 30)

    x, y = np.arange(n), np.arange(n)
    X, Y = np.meshgrid(x, y)
    blob = amplitude * np.exp(-((X - x0) ** 2 + (Y - y0) ** 2) / (2 * sigma ** 2))
    data += blob

data /= np.max(np.abs(data))

## Linear vs power-law normalization

`power_norm` raises values to a power `gamma < 1` (preserving sign), which compresses the near-zero range and stretches values near the extremes across more of the colormap. `gamma=0.5` (square root) is a mild, smooth compromise; lower values push harder towards the extremes but start making background noise visible as speckle (see the comparison further down).

In [ ]:
from aule.plots._style import power_norm

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

im0 = axes[0].imshow(data, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title('Linear normalization')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(data, cmap='RdBu_r', norm=power_norm(data, gamma=0.5))
axes[1].set_title('Power-law normalization (gamma=0.5)')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(data, cmap='RdBu_r', norm=power_norm(data, gamma=0.3))
axes[2].set_title('Power-law normalization (gamma=0.3, stronger)')
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()

Notice `gamma=0.3` pushes the colormap so hard towards the extremes that the background noise becomes visible as speckle — a smaller `gamma` is not strictly better. `gamma=0.5` is usually a safer default; tune it down gradually and stop as soon as speckle starts to dominate.

## Symmetric-log normalization

`symlog_norm` keeps a small linear region around zero (set via `linthresh`) where background noise stays calm, and switches to a logarithmic scale beyond it — spreading the colormap so two different strong anomalies actually look different, instead of both saturating to the same dark color. This usually gives the *strongest* extreme-value contrast of the available options, at the cost of making near-threshold fluctuations show up as speckle outside the linear zone.

In [ ]:
from aule.plots._style import symlog_norm

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, linthresh in zip(axes, [0.01, 0.03, 0.08]):
    norm = symlog_norm(data, linthresh=linthresh, linscale=0.5)
    im = ax.imshow(data, cmap='RdBu_r', norm=norm)
    ax.set_title(f'SymLog normalization (linthresh={linthresh})')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

`linthresh=0.01` is too tight (most of the noise sits outside the linear zone and shows up as speckle); `linthresh=0.08` is too wide (barely different from linear). `linthresh=0.03` here is the sweet spot: background noise stays quiet, extremes get clearly separated colors. If you don't want to tune this by hand, leaving `linthresh=None` makes `symlog_norm` pick it automatically as the 75th percentile of `|data|`.

In [ ]:
auto_norm = symlog_norm(data)  # linthresh inferred automatically
print('Automatically chosen linthresh:', auto_norm.linthresh)

## Two-slope normalization for asymmetric data

The normalizations above assume data that's roughly symmetric around zero. If your real data is skewed — say, mostly small positive anomalies with a few much larger negative ones — a plain symmetric range wastes colormap resolution on the side with less signal and clips the other. `asymmetric_twoslope_norm` anchors a chosen `vcenter` (not necessarily 0) to the colormap's midpoint, with independent scaling on either side.

In [ ]:
from aule.plots._style import asymmetric_twoslope_norm

# skewed data: mostly small positive noise, a few large negative dips
skewed = np.random.exponential(0.15, (n, n)) - 0.3
for _ in range(8):
    x0, y0 = np.random.randint(0, n, size=2)
    sigma = np.random.uniform(10, 25)
    x, y = np.arange(n), np.arange(n)
    X, Y = np.meshgrid(x, y)
    skewed -= 1.2 * np.exp(-((X - x0) ** 2 + (Y - y0) ** 2) / (2 * sigma ** 2))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

vmax = float(np.max(np.abs(skewed)))
im0 = axes[0].imshow(skewed, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title('Symmetric linear (wastes range on the small side)')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(skewed, cmap='RdBu_r', norm=asymmetric_twoslope_norm(skewed, vcenter=0.0))
axes[1].set_title('Two-slope (uses full data range on each side)')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

## Using these directly in aule's spatial plots

`plot_bias_map`, `plot_error_map` (signed branch), and `plot_field_comparison` (difference panel) all expose a `norm_type` parameter (`"linear"`, `"power"`, `"symlog"`, or `"twoslope"`) plus a `norm_kwargs` dict forwarded to the chosen normalization — no need to build the normalization object by hand for everyday use.

In [ ]:
from aule.plots import plot_bias_map

gt   = np.random.rand(8, 64, 64, 1)
pred = gt + np.random.normal(0, 0.1, gt.shape)

fig, ax = plot_bias_map(gt, pred, norm_type='symlog', norm_kwargs={'linthresh': 0.02})

In [ ]:
from aule.plots import plot_error_map

fig, ax = plot_error_map(gt[0], pred[0], abs_error=False, norm_type='power', norm_kwargs={'gamma': 0.4})

## Quick recommendation

For everyday bias/error maps, start with the default `"linear"`. If extremes look washed out and your data is roughly symmetric around zero, try `"power"` with `gamma` around 0.4-0.5 first (smooth, predictable); reach for `"symlog"` when you specifically need to distinguish *different strengths* of extreme values, not just see that they exist. Use `"twoslope"` whenever the data itself is asymmetric around its center, regardless of which of the above you'd otherwise pick for the extreme-contrast behavior.